# Synthetic Data Vault API Tutorial

This notebook is a short API-style tutorial for the main SDV functions used in this project.

The goal is to show the basic workflow of SDV in a simple and readable way:

1. Create metadata for a single table.
2. Fit a synthesizer on real data.
3. Sample synthetic data.
4. Evaluate synthetic data quality.

This notebook is intentionally smaller than the main project notebook. The full experiment is available in `Synthetic_Data_Vault.ipynb`.


## 1. Import Required Libraries

SDV is used for synthetic data generation, while SDMetrics is used to evaluate synthetic data quality.


In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer
from sdmetrics.reports.single_table import DiagnosticReport, QualityReport

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Imports completed successfully.")

## 2. Load and Clean a Small Example Dataset

For the API demo, I use a small sample of the Adult Income dataset so the notebook runs quickly.


In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)
df = adult.frame.copy()

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace("-", "_")
    .str.replace(" ", "_")
)

df = df.replace("?", np.nan)
df = df.dropna().reset_index(drop=True)

for column in df.columns:
    if str(df[column].dtype) == "category":
        df[column] = df[column].astype(str)

target_col = "class"
df[target_col] = df[target_col].astype(str).str.strip()

# Use a small sample to keep the API tutorial fast.
df_sample = df.sample(n=3000, random_state=RANDOM_STATE).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
df_sample.head()

## 3. Detect Metadata

SDV needs metadata to understand the structure of the table. The metadata stores information about column names and data types.


In [ ]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df_sample)

print("Metadata detected successfully.")
metadata.to_dict()

## 4. Fit the Gaussian Copula Synthesizer

The Gaussian Copula synthesizer learns patterns from the real data and then generates new synthetic rows.


In [ ]:
synthesizer = GaussianCopulaSynthesizer(
    metadata,
    enforce_min_max_values=True,
    enforce_rounding=True
)

synthesizer.fit(df_sample)

print("Synthesizer fitted successfully.")

## 5. Sample Synthetic Data

After fitting, the synthesizer can generate new rows with similar structure and statistical patterns.


In [ ]:
synthetic_sample = synthesizer.sample(num_rows=1000)

print("Synthetic sample shape:", synthetic_sample.shape)
synthetic_sample.head()

## 6. Evaluate Synthetic Data Quality

The Diagnostic Report checks whether the synthetic data follows expected structural rules.  
The Quality Report checks whether the synthetic data is statistically similar to the real data.


In [ ]:
metadata_dict = metadata.to_dict()

diagnostic_report = DiagnosticReport()
diagnostic_report.generate(
    real_data=df_sample,
    synthetic_data=synthetic_sample,
    metadata=metadata_dict,
    verbose=False
)

print("Diagnostic score:", diagnostic_report.get_score())

In [ ]:
quality_report = QualityReport()
quality_report.generate(
    real_data=df_sample,
    synthetic_data=synthetic_sample,
    metadata=metadata_dict,
    verbose=False
)

print("Quality score:", quality_report.get_score())
quality_report.get_properties()

## API Tutorial Summary

This notebook demonstrated the basic SDV API workflow:

1. Detect metadata from a real table.
2. Fit a synthetic data generator.
3. Generate synthetic rows.
4. Evaluate the synthetic data.

The main project notebook builds on this API workflow by adding EDA, model training, real-vs-synthetic comparison, and result interpretation.
